# Fine-tune the wall model on our own plans**The problem this solves.** The wall model is trained on CubiCasa5K — Finnish plans inone house style. Ours are UK estate-agent plans in twenty styles, and the gap shows upexactly where you noticed it: door swing arcs are suppressed on some plans and leftstanding on others. Measured across the arcs we could detect, the model wipes out 42% ofthem completely, leaves 8% completely intact, and half-erases the rest. Inconsistency isworse than either extreme, because nothing downstream can tell which it got.**Why fine-tuning and not more rules.** We tried the rule. A geometric door-swing detector— fit a circle, check the radius is a door width, check it spans a quadrant — found 12arcs across 25 plans. It missed almost everything, because agents draw swings a dozendifferent ways: quarter circles, straight leaf lines, thin light strokes over a colourfill, or nothing at all. Every plan style needs its own rule. That is what a model is for.**Before you start**, on your own machine:```bashpython -m tools.annotate_walls          # correct 20-30 plans, a minute or two eachpython -m tools.annotate_walls --export # writes data/golden/wall_training/```You are correcting, not tracing: each plan opens with the model's own reading painted on,and the job is scrubbing off the door swings and cabinet runs it wrongly called wall.Then zip `data/golden/wall_training/` and upload it in §3.**Runtime → Change runtime type → T4 GPU.** About 40 minutes.

## 1. Setup

In [ ]:
import torch, subprocessprint(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"%pip install -q segmentation-models-pytorch safetensors albumentationsimport segmentation_models_pytorch as smp, albumentations as Aprint("smp", smp.__version__, "| albumentations", A.__version__)

## 2. The starting weightsWe fine-tune *from* the CubiCasa checkpoint rather than from ImageNet. It already knowswhat a wall is in general; what it does not know is our drawing conventions, and that is amuch smaller thing to learn than the whole task.

In [ ]:
from pathlib import Pathimport urllib.requestimport torch, segmentation_models_pytorch as smpfrom safetensors.torch import load_file, save_fileBASE = Path("/content/plan_walls_base.safetensors")URL = "https://huggingface.co/Yytsi/floorplan-to-3d-walls/resolve/main/best.safetensors"if not BASE.exists():    urllib.request.urlretrieve(URL, BASE)    print(f"downloaded {BASE.stat().st_size/1e6:.0f} MB")def fresh_model():    m = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=4)    m.load_state_dict(load_file(BASE))    return mprint("baseline loaded")

## 3. Your corrected plansUpload the zip of `data/golden/wall_training/`.Your masks are binary — wall or not — while the model has four classes (floor, wall, door,window). We keep all four heads and supervise only the wall/not-wall distinction, byfolding door and window into wall for the loss. That is honest about what you labelled,and it leaves the door and window heads free rather than teaching them something false.

In [ ]:
import zipfile, json, shutilfrom pathlib import Pathfrom google.colab import filesTRAIN = Path("/content/wall_training")if TRAIN.exists():    shutil.rmtree(TRAIN)TRAIN.mkdir(parents=True)print("choose the zip of data/golden/wall_training/ …")up = files.upload()name = next(iter(up))with zipfile.ZipFile(name) as z:    z.extractall(TRAIN)# the zip may or may not contain the wrapping directoryroot = TRAIN if (TRAIN / "manifest.json").exists() else next(TRAIN.glob("*/manifest.json")).parentmanifest = json.loads((root / "manifest.json").read_text())print(f"{manifest['count']} corrected plans")if manifest["count"] < 15:    print("WARNING: under 15 plans. Expect overfitting — the held-out numbers in §7 "          "will tell you whether it happened, but more labels is the real answer.")

## 4. Hold some out, honestlyTwo plans is not a validation set, so this splits by a hash of the listing id rather thanby shuffling: rerunning the notebook, or adding more labels later, never moves a plan fromone side to the other. The same rule the project's frozen holdout uses.

In [ ]:
import hashlibdef side(listing_id, holdout_fraction=0.25):    h = int(hashlib.sha256(listing_id.encode()).hexdigest()[:8], 16)    return "val" if (h % 1000) / 1000.0 < holdout_fraction else "train"split = {"train": [], "val": []}for e in manifest["entries"]:    split[side(e["listing_id"])].append(e)print(f"{len(split['train'])} train, {len(split['val'])} val")print("val plans:", " ".join(e["listing_id"] for e in split["val"]))if not split["val"]:    print("no validation plans — with this few labels, treat §7 as unmeasured")

## 5. The dataTwo things make a small label set go further.**Crops, not whole plans.** A plan resized to 512 is one training example; 512-pixel cropsof it at native resolution are dozens, each showing walls at the thickness the model willmeet at inference. Twenty plans becomes a few thousand crops.**Augmentation aimed at the actual variation.** Our plans differ from CubiCasa5K incolour, contrast, line weight and orientation — not in content — so the augmentationstarget exactly that and nothing else. No flips that would produce mirror-image plans noagent draws.

In [ ]:
import numpy as np, cv2, albumentations as Afrom PIL import Imagefrom torch.utils.data import Dataset, DataLoaderimport torchCROP = 512MEAN = np.array([0.485, 0.456, 0.406], np.float32)STD = np.array([0.229, 0.224, 0.225], np.float32)train_aug = A.Compose([    A.RandomScale(scale_limit=(-0.4, 0.3), p=0.8),        # walls thick and thin    A.PadIfNeeded(CROP, CROP, border_mode=cv2.BORDER_CONSTANT, value=255, mask_value=0),    A.RandomCrop(CROP, CROP),    A.Rotate(limit=6, border_mode=cv2.BORDER_CONSTANT, value=255, mask_value=0, p=0.4),    A.RandomBrightnessContrast(0.25, 0.35, p=0.7),        # faded scans, heavy prints    A.HueSaturationValue(18, 30, 12, p=0.5),              # the colour-filled plans    A.OneOf([A.GaussNoise(p=1), A.ImageCompression(p=1)], p=0.3),   # scan grain, JPEG mush    A.Sharpen(p=0.2),])val_aug = A.Compose([    A.PadIfNeeded(CROP, CROP, border_mode=cv2.BORDER_CONSTANT, value=255, mask_value=0),    A.CenterCrop(CROP, CROP),])class Plans(Dataset):    '''Crops of our corrected plans. repeat multiplies epochs-worth per plan.'''    def __init__(self, entries, root, aug, repeat=1):        self.items = []        for e in entries:            img = np.array(Image.open(root / e["image"]).convert("RGB"))            m = np.array(Image.open(root / e["mask"]).convert("L"))            self.items.append((img, (m > 127).astype(np.uint8)))        self.aug = aug        self.repeat = repeat    def __len__(self):        return len(self.items) * self.repeat    def __getitem__(self, i):        img, m = self.items[i % len(self.items)]        out = self.aug(image=img, mask=m)        x = (out["image"].astype(np.float32) / 255.0 - MEAN) / STD        return (torch.from_numpy(x).permute(2, 0, 1).float(),                torch.from_numpy(out["mask"]).long())train_ds = Plans(split["train"], root, train_aug, repeat=48)val_ds = Plans(split["val"], root, val_aug, repeat=4) if split["val"] else Nonetrain_dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, drop_last=True)val_dl = DataLoader(val_ds, batch_size=8, num_workers=2) if val_ds else Noneprint(f"{len(train_ds)} training crops, {len(val_ds) if val_ds else 0} validation crops")

## 6. Train**The loss is the interesting part.** Your masks say wall or not-wall; the model has fourclasses. So we collapse its floor/wall/door/window logits into two — wall-ish (wall, door,window) against floor — and supervise that. The model keeps its four heads and its abilityto tell a door from a window; it only learns *where* to draw the line between structure andeverything else, which is exactly the thing that is wrong.**A low learning rate and a frozen encoder.** We are correcting a bias, not teaching thetask. The encoder already knows what plan drawings look like; letting it move on twentyimages is how you destroy a good model.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as Ffrom tqdm.auto import tqdmdevice = "cuda"model = fresh_model().to(device)for p in model.encoder.parameters():          # keep what it already knows    p.requires_grad = FalseEPOCHS = 8opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],                        lr=2e-4, weight_decay=1e-4)sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4,                                            total_steps=EPOCHS * len(train_dl))scaler = torch.amp.GradScaler("cuda")def binary_logits(logits):    '''floor vs (wall|door|window), as a 2-class problem the labels can supervise.'''    floor = logits[:, 0:1]    structure = logits[:, 1:].logsumexp(dim=1, keepdim=True)    return torch.cat([floor, structure], dim=1)def dice(logits2, target, eps=1.0):    prob = logits2.softmax(1)[:, 1]    t = target.float()    inter = (prob * t).sum((1, 2))    return 1 - ((2 * inter + eps) / (prob.sum((1, 2)) + t.sum((1, 2)) + eps)).mean()def step_loss(logits, y):    b = binary_logits(logits)    # walls are a few percent of the page, so weight them up and add Dice, which    # does not care about the imbalance at all    w = torch.tensor([1.0, 4.0], device=logits.device)    return F.cross_entropy(b, y, weight=w) + dice(b, y)@torch.no_grad()def evaluate(dl):    model.eval()    inter = union = 0.0    for x, y in dl:        x, y = x.to(device), y.to(device)        pred = binary_logits(model(x)).argmax(1)        inter += ((pred == 1) & (y == 1)).sum().item()        union += ((pred == 1) | (y == 1)).sum().item()    model.train()    return inter / max(union, 1)if val_dl:    print(f"wall IoU before fine-tuning: {evaluate(val_dl):.3f}")history = []for epoch in range(EPOCHS):    running = 0.0    for x, y in tqdm(train_dl, desc=f"epoch {epoch+1}/{EPOCHS}", leave=False):        x, y = x.to(device), y.to(device)        opt.zero_grad(set_to_none=True)        with torch.amp.autocast("cuda"):            loss = step_loss(model(x), y)        scaler.scale(loss).backward()        scaler.step(opt); scaler.update(); sched.step()        running += loss.item()    iou = evaluate(val_dl) if val_dl else float("nan")    history.append({"epoch": epoch + 1, "loss": running / len(train_dl), "val_wall_iou": iou})    print(f"epoch {epoch+1}  loss {history[-1]['loss']:.4f}  val wall IoU {iou:.3f}")

## 7. Did it work?Two questions, and the second one matters more.**Did wall IoU improve on the held-out plans?** If it went *down*, you have too few labelsand the model has memorised them — stop, label more, come back.**Did it forget?** A model that scores brilliantly on twenty UK plans and has lost what itknew about everything else is worse than the one you started with. So we also check a planit never saw *and* eyeball the before-and-after directly.

In [ ]:
import matplotlib.pyplot as pltif val_dl:    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))    ax[0].plot([h["epoch"] for h in history], [h["loss"] for h in history], marker="o")    ax[0].set_title("training loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)    ax[1].plot([h["epoch"] for h in history], [h["val_wall_iou"] for h in history],               marker="o", color="seagreen")    ax[1].set_title("held-out wall IoU"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3)    plt.tight_layout(); plt.show()    best = max(h["val_wall_iou"] for h in history)    print(f"best held-out wall IoU: {best:.3f}")

In [ ]:
import numpy as np, cv2from PIL import ImagePALETTE = np.array([[250, 250, 248], [25, 32, 48], [220, 90, 60], [70, 140, 220]], np.uint8)def predict_full(net, rgb, size=512):    '''Letterbox to 512, predict, unpad -- the same path inference uses.'''    h, w = rgb.shape[:2]    s = size / max(h, w)    nh, nw = max(1, int(h * s)), max(1, int(w * s))    canvas = np.full((size, size, 3), 255, np.uint8)    top, left = (size - nh) // 2, (size - nw) // 2    canvas[top:top+nh, left:left+nw] = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)    x = (canvas.astype(np.float32) / 255.0 - MEAN) / STD    net.eval()    with torch.no_grad():        out = net(torch.from_numpy(x).permute(2, 0, 1)[None].float().to(device))    pred = out[0].argmax(0).cpu().numpy().astype(np.uint8)[top:top+nh, left:left+nw]    return cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)baseline = fresh_model().to(device)show = split["val"] or split["train"][:3]for e in show[:4]:    rgb = np.array(Image.open(root / e["image"]).convert("RGB"))    truth = np.array(Image.open(root / e["mask"]).convert("L")) > 127    fig, axes = plt.subplots(1, 4, figsize=(19, 5))    for ax, img, title in zip(            axes,            [rgb, np.where(truth[..., None], PALETTE[1], PALETTE[0]),             PALETTE[predict_full(baseline, rgb)], PALETTE[predict_full(model, rgb)]],            [f"{e['listing_id']} — the plan", "your correction",             "before fine-tuning", "after fine-tuning"]):        ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis("off")    plt.tight_layout(); plt.show()

## 8. Save itSame filename and layout as the model it replaces, so dropping it into `models/` is thewhole installation. `pipeline/floorplan/wallnet.py` needs no change.

In [ ]:
from safetensors.torch import save_filefrom google.colab import filesimport jsonmodel.eval()state = {k: v.cpu().contiguous() for k, v in model.state_dict().items()}save_file(state, "/content/plan_walls.safetensors")note = {    "base": "Yytsi/floorplan-to-3d-walls (CubiCasa5K, MIT)",    "fine_tuned_on": f"{len(split['train'])} hand-corrected UK estate-agent plans",    "held_out": [e["listing_id"] for e in split["val"]],    "epochs": EPOCHS,    "encoder": "frozen",    "supervision": "binary wall vs floor, folded from the 4-class head",    "history": history,}Path("/content/plan_walls.json").write_text(json.dumps(note, indent=1))files.download("/content/plan_walls.safetensors")files.download("/content/plan_walls.json")

## 9. Install and checkOn your own machine:```bashmv ~/Downloads/plan_walls.safetensors models/plan_walls.safetensorsmv ~/Downloads/plan_walls.json      models/plan_walls.jsonpython -m pipeline run $(python -c "import json;print(' '.join(l['listing_id'] for l in json.load(open('data/golden/golden_set.json'))['listings']))") --from 5python -m tools.plan_vs_shell build --out out/review```Then read `out/review/plan-vs-shell.html`. The number to watch is **outline on walls** —0.827 before fine-tuning. And look at the door swings on the plans you *did not* label:those are the ones that tell you whether it generalised or memorised.If it got worse, `python -m tools.fetch_wallnet --force` puts the original back.**One caveat about these labels.** They were seeded from the model's own reading andcorrected, so they are not independent of it: places where the model is confidently wrongin a way that looks plausible are places you were less likely to fix. That is fine forclosing a known gap like door swings, and it is not a substitute for tracing a handful ofplans from scratch if you ever want a clean measurement.